In [1]:
ADAPTER_PATH = "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"
TEST_GENERATION = True

In [2]:
import shutil

shutil.copytree(
    ADAPTER_PATH,
    "/kaggle/working/",
    dirs_exist_ok=True,
)

shutil.copytree(
    "/kaggle/input/notebooks/huikang/nvidia-nemotron-all-linear",
    "/kaggle/working/reference",
    dirs_exist_ok=True,
)

'/kaggle/working/reference'

In [3]:
import zipfile

with zipfile.ZipFile("reference/submission.zip", "r") as zip_ref:
    zip_ref.extractall("reference")

# Compare configs

In [4]:
import json

with open("reference/adapter_config.json") as f:
    reference_adapter_config = json.load(f)

print(reference_adapter_config)

{'alora_invocation_tokens': None, 'alpha_pattern': {}, 'arrow_config': None, 'auto_mapping': None, 'base_model_name_or_path': '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', 'bias': 'none', 'corda_config': None, 'ensure_weight_tying': False, 'eva_config': None, 'exclude_modules': None, 'fan_in_fan_out': False, 'inference_mode': True, 'init_lora_weights': True, 'layer_replication': None, 'layers_pattern': None, 'layers_to_transform': None, 'loftq_config': {}, 'lora_alpha': 16, 'lora_bias': False, 'lora_dropout': 0.05, 'megatron_config': None, 'megatron_core': 'megatron.core', 'modules_to_save': None, 'peft_type': 'LORA', 'peft_version': '0.18.1', 'qalora_group_size': 16, 'r': 32, 'rank_pattern': {}, 'revision': None, 'target_modules': ['k_proj', 'o_proj', 'in_proj', 'q_proj', 'up_proj', 'v_proj', 'down_proj', 'out_proj'], 'target_parameters': None, 'task_type': 'CAUSAL_LM', 'trainable_token_indices': None, 'use_dora': False, 'use_qalora': False, 'use_r

In [5]:
import json

with open("adapter_config.json") as f:
    trained_adapter_config = json.load(f)

print(trained_adapter_config)

{'alpha_pattern': {}, 'auto_mapping': None, 'base_model_name_or_path': None, 'bias': 'none', 'corda_config': None, 'eva_config': None, 'exclude_modules': None, 'fan_in_fan_out': False, 'inference_mode': False, 'init_lora_weights': True, 'layer_replication': None, 'layers_pattern': None, 'layers_to_transform': None, 'loftq_config': {}, 'lora_alpha': 32, 'lora_bias': False, 'lora_dropout': 0, 'megatron_config': None, 'megatron_core': 'megatron.core', 'modules_to_save': None, 'peft_type': 'LORA', 'r': 32, 'rank_pattern': {}, 'revision': None, 'target_modules': 'all-linear', 'task_type': 'CAUSAL_LM', 'trainable_token_indices': None, 'use_dora': False, 'use_rslora': False}


In [6]:
for k, reference_value in reference_adapter_config.items():
    if k in trained_adapter_config and reference_value != trained_adapter_config[k]:
        print(k)
        print(reference_value)
        print(trained_adapter_config[k])
        print()

base_model_name_or_path
/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
None

inference_mode
True
False

lora_alpha
16
32

lora_dropout
0.05
0

target_modules
['k_proj', 'o_proj', 'in_proj', 'q_proj', 'up_proj', 'v_proj', 'down_proj', 'out_proj']
all-linear



In [7]:
for k, reference_value in reference_adapter_config.items():
    if k not in trained_adapter_config:
        print(k)
        print(reference_value)
        print()

alora_invocation_tokens
None

arrow_config
None

ensure_weight_tying
False

peft_version
0.18.1

qalora_group_size
16

target_parameters
None

use_qalora
False



In [8]:
for k, v in trained_adapter_config.items():
    if k not in reference_adapter_config:
        print(k)

# Align configs

In [9]:
trained_adapter_config["target_modules"] = [
    "k_proj",
    "o_proj",
    "in_proj",
    "q_proj",
    "up_proj",
    "v_proj",
    "down_proj",
    "out_proj",
    "lm_head",
]

with open("adapter_config.json", "w") as f:
    f.write(json.dumps(trained_adapter_config))

In [10]:
with open("adapter_config.json") as f:
    print(f.read())

{"alpha_pattern": {}, "auto_mapping": null, "base_model_name_or_path": null, "bias": "none", "corda_config": null, "eva_config": null, "exclude_modules": null, "fan_in_fan_out": false, "inference_mode": false, "init_lora_weights": true, "layer_replication": null, "layers_pattern": null, "layers_to_transform": null, "loftq_config": {}, "lora_alpha": 32, "lora_bias": false, "lora_dropout": 0, "megatron_config": null, "megatron_core": "megatron.core", "modules_to_save": null, "peft_type": "LORA", "r": 32, "rank_pattern": {}, "revision": null, "target_modules": ["k_proj", "o_proj", "in_proj", "q_proj", "up_proj", "v_proj", "down_proj", "out_proj", "lm_head"], "task_type": "CAUSAL_LM", "trainable_token_indices": null, "use_dora": false, "use_rslora": false}


# Compare adapters

In [11]:
def trained_adapter_key_rename(key_name: str) -> str:
    key_name = key_name.replace("base_model.model.model", "base_model.model.backbone")
    return key_name

In [12]:
from safetensors import safe_open

trained_adapter_keys = set()
with safe_open("adapter_model.safetensors", framework="pt", device="cpu") as f:
    for key in f.keys():
        tensor_slice = f.get_slice(key)
        trained_adapter_keys.add(
            (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
        )

In [13]:
from safetensors import safe_open

reference_adapter_keys = set()
with safe_open(
    "reference/adapter_model.safetensors", framework="pt", device="cpu"
) as f:
    for key in f.keys():
        tensor_slice = f.get_slice(key)
        reference_adapter_keys.add(
            (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
        )

In [14]:
from safetensors import safe_open
import glob

model_keys = set()
for model_safetensors in glob.glob(
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1/*.safetensors"
):
    with safe_open(model_safetensors, framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor_slice = f.get_slice(key)
            model_keys.add(
                (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
            )

In [15]:
# set([".".join(x.split(".")[3:]) for x in trained_adapter_keys]) - set([".".join(x.split(".")[3:]) for x in reference_adapter_keys])

In [16]:
sorted(model_keys)[-20:]

[('backbone.layers.8.mixer.experts.98.down_proj.weight', (2688, 1856), 'BF16'),
 ('backbone.layers.8.mixer.experts.98.up_proj.weight', (1856, 2688), 'BF16'),
 ('backbone.layers.8.mixer.experts.99.down_proj.weight', (2688, 1856), 'BF16'),
 ('backbone.layers.8.mixer.experts.99.up_proj.weight', (1856, 2688), 'BF16'),
 ('backbone.layers.8.mixer.gate.e_score_correction_bias', (128,), 'F32'),
 ('backbone.layers.8.mixer.gate.weight', (128, 2688), 'BF16'),
 ('backbone.layers.8.mixer.shared_experts.down_proj.weight',
  (2688, 3712),
  'BF16'),
 ('backbone.layers.8.mixer.shared_experts.up_proj.weight',
  (3712, 2688),
  'BF16'),
 ('backbone.layers.8.norm.weight', (2688,), 'BF16'),
 ('backbone.layers.9.mixer.A_log', (64,), 'F32'),
 ('backbone.layers.9.mixer.D', (64,), 'F32'),
 ('backbone.layers.9.mixer.conv1d.bias', (6144,), 'BF16'),
 ('backbone.layers.9.mixer.conv1d.weight', (6144, 1, 4), 'BF16'),
 ('backbone.layers.9.mixer.dt_bias', (64,), 'BF16'),
 ('backbone.layers.9.mixer.in_proj.weight', (1

In [17]:
sorted(reference_adapter_keys)[-20:]

[('base_model.model.backbone.layers.8.mixer.experts.97.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_pro

In [18]:
sorted(trained_adapter_keys)[-20:]

[('base_model.model.model.layers.7.mixer.x_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.model.layers.7.mixer.x_proj.lora_B.weight',
  (4096, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w1.lora_A.weight',
  (1, 32, 2688),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w1.lora_B.weight',
  (128, 1856, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w2.lora_A.weight',
  (128, 32, 1856),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w2.lora_B.weight',
  (1, 2688, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w3.lora_A.weight',
  (0,),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w3.lora_B.weight',
  (0,),
  'F32'),
 ('base_model.model.model.layers.8.mixer.shared_experts.down_proj.lora_A.weight',
  (32, 3712),
  'F32'),
 ('base_model.model.model.layers.8.mixer.shared_experts.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.shared_experts.

In [19]:
len(trained_adapter_keys), len(reference_adapter_keys)

(418, 12008)

In [20]:
(
    len(trained_adapter_keys - reference_adapter_keys),
    len(reference_adapter_keys - trained_adapter_keys),
)

(418, 12008)

# Update adapter

In [21]:
import re

import torch
from safetensors import safe_open
from safetensors.torch import save_file

# --- Load all trained adapter tensors ---
adapter_tensors = {}
with safe_open("adapter_model.safetensors", framework="pt", device="cpu") as f:
    for key in f.keys():
        adapter_tensors[key] = f.get_tensor(key)

# --- Collect adapter base names (without .lora_A/.lora_B.weight suffix) ---
base_names = set()
for key in adapter_tensors:
    base = re.sub(r"\.lora_[AB]\.weight$", "", key)
    base_names.add(base)

# --- Identify Mamba layers needing gate_proj+x_proj → in_proj ---
mamba_merge_layers = {}  # layer_path -> {"gate_proj": base, "x_proj": base}
for base in base_names:
    for proj in ("gate_proj", "x_proj"):
        if f".{proj}" in base:
            layer_path = base.rsplit(f".{proj}", 1)[0]
            mamba_merge_layers.setdefault(layer_path, {})[proj] = base
mamba_merge_bases = set()
for projs in mamba_merge_layers.values():
    mamba_merge_bases.update(projs.values())

# --- Build model_key_shapes for in_proj dimension lookup ---
model_key_shapes = {k: s for k, s, _ in model_keys}

# --- Build output tensors ---
tensors = {}

for base in sorted(base_names):
    lora_A = adapter_tensors[f"{base}.lora_A.weight"]
    lora_B = adapter_tensors[f"{base}.lora_B.weight"]
    renamed = trained_adapter_key_rename(base)

    # Skip empty w3 experts
    if ".experts.w3" in base and lora_A.numel() == 0:
        continue

    # # Skip lm_head (not in reference adapter)
    # if ".lm_head" in base:
    #     continue

    # Skip gate_proj/x_proj — handled in Mamba merge pass below
    if base in mamba_merge_bases:
        continue

    # --- Expert unfusing: w1 → per-expert up_proj, w2 → per-expert down_proj ---
    if ".experts.w1" in base or ".experts.w2" in base:
        # Broadcast shared dimension (one of A/B has shape[0]==1)
        # Use expand + contiguous to avoid shared memory in safetensors
        if lora_A.shape[0] == 1:
            lora_A = lora_A.expand(lora_B.shape[0], -1, -1).contiguous()
        elif lora_B.shape[0] == 1:
            lora_B = lora_B.expand(lora_A.shape[0], -1, -1).contiguous()

        num_experts = lora_A.shape[0]
        proj_name = "up_proj" if ".w1" in base else "down_proj"

        for i in range(num_experts):
            exp_renamed = re.sub(
                r"\.experts\.w[12]",
                f".experts.{i}.{proj_name}",
                renamed,
            )
            tensors[f"{exp_renamed}.lora_A.weight"] = lora_A[i].contiguous()
            tensors[f"{exp_renamed}.lora_B.weight"] = lora_B[i].contiguous()
        continue

    # --- Direct rename for everything else ---
    tensors[f"{renamed}.lora_A.weight"] = lora_A
    tensors[f"{renamed}.lora_B.weight"] = lora_B

# --- Mamba: gate_proj + x_proj → in_proj via SVD ---
for layer_path, projs in sorted(mamba_merge_layers.items()):
    renamed_layer = trained_adapter_key_rename(layer_path)
    in_proj_base = f"{renamed_layer}.in_proj"

    model_in_proj_key = (
        renamed_layer.replace("base_model.model.", "") + ".in_proj.weight"
    )
    in_proj_dim = model_key_shapes[model_in_proj_key][0]

    gate_A = adapter_tensors[f"{projs['gate_proj']}.lora_A.weight"].float()
    gate_B = adapter_tensors[f"{projs['gate_proj']}.lora_B.weight"].float()
    x_A = adapter_tensors[f"{projs['x_proj']}.lora_A.weight"].float()
    x_B = adapter_tensors[f"{projs['x_proj']}.lora_B.weight"].float()
    rank = gate_A.shape[0]

    # Build combined rank-64 representation, then SVD to best rank-32
    A_cat = torch.cat([gate_A, x_A], dim=0)  # (64, in_dim)
    B_block = torch.zeros(in_proj_dim, 2 * rank)
    B_block[: gate_B.shape[0], :rank] = gate_B
    B_block[gate_B.shape[0] : gate_B.shape[0] + x_B.shape[0], rank:] = x_B

    Q_B, R_B = torch.linalg.qr(B_block)
    Q_A, R_A = torch.linalg.qr(A_cat.T)
    core = R_B @ R_A.T
    U, S, Vh = torch.linalg.svd(core, full_matrices=False)

    k = rank
    new_B = (Q_B @ U[:, :k]) * S[:k].unsqueeze(0)
    new_A = Vh[:k, :] @ Q_A.T

    kept = S[:k].sum().item()
    total = S.sum().item()
    print(
        f"{layer_path}: SVD kept {kept:.2f}/{total:.2f} "
        f"({kept / total * 100:.1f}%) of singular value mass"
    )

    tensors[f"{in_proj_base}.lora_A.weight"] = new_A
    tensors[f"{in_proj_base}.lora_B.weight"] = new_B

print(
    f"\nConverted {len(adapter_tensors)} trained tensors → {len(tensors)} output tensors"
)
save_file(tensors, "adapter_model.safetensors")

base_model.model.model.layers.0.mixer: SVD kept 4.48/5.94 (75.4%) of singular value mass
base_model.model.model.layers.11.mixer: SVD kept 3.93/5.22 (75.2%) of singular value mass
base_model.model.model.layers.14.mixer: SVD kept 3.98/5.31 (75.0%) of singular value mass
base_model.model.model.layers.16.mixer: SVD kept 4.06/5.39 (75.3%) of singular value mass
base_model.model.model.layers.18.mixer: SVD kept 3.95/5.25 (75.2%) of singular value mass
base_model.model.model.layers.2.mixer: SVD kept 3.98/5.09 (78.1%) of singular value mass
base_model.model.model.layers.21.mixer: SVD kept 4.04/5.35 (75.4%) of singular value mass
base_model.model.model.layers.23.mixer: SVD kept 4.15/5.50 (75.4%) of singular value mass
base_model.model.model.layers.25.mixer: SVD kept 4.20/5.66 (74.3%) of singular value mass
base_model.model.model.layers.28.mixer: SVD kept 4.58/5.97 (76.8%) of singular value mass
base_model.model.model.layers.30.mixer: SVD kept 4.78/6.26 (76.5%) of singular value mass
base_model.m

In [22]:
from safetensors import safe_open

updated_adapter_keys = set()
with safe_open("adapter_model.safetensors", framework="pt", device="cpu") as f:
    for key in f.keys():
        tensor_slice = f.get_slice(key)
        updated_adapter_keys.add(
            (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
        )

In [23]:
sorted(updated_adapter_keys)[-20:]

[('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.up_proj.lo

In [24]:
len(updated_adapter_keys), len(reference_adapter_keys)

(12010, 12008)

In [25]:
(
    len(updated_adapter_keys - reference_adapter_keys),
    len(reference_adapter_keys - updated_adapter_keys),
)

(2, 0)

In [26]:
sorted(reference_adapter_keys - updated_adapter_keys)[:20]

[]

In [27]:
sorted(updated_adapter_keys - reference_adapter_keys)[:20]

[('base_model.model.backbone.lm_head.lora_A.weight', (32, 2688), 'F32'),
 ('base_model.model.backbone.lm_head.lora_B.weight', (131072, 32), 'F32')]

# Load model

In [28]:
"""Metric for NVIDIA (129716)."""

import subprocess
import sys

# Set up environment
commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]
if TEST_GENERATION:
    for cmd in commands:
        print(f"Running: {cmd}")
        subprocess.run(cmd, shell=True, check=True)
sys.path.insert(0, "/tmp")

Running: uv pip uninstall torch torchvision torchaudio


Using Python 3.12.12 environment at: /usr
Uninstalled 3 packages in 797ms
 - torch==2.9.0+cu126
 - torchaudio==2.9.0+cu126
 - torchvision==0.24.0+cu126


Running: tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell


In [29]:
import glob
import math
import multiprocessing
import os
import re
import time
from pathlib import Path

import kagglehub
import pandas as pd
from tqdm import tqdm

# Configuration
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
DATA_PATH = (
    ""  # Path(kagglehub.dataset_download('metric/nvidia-nemotron-rerun-data-129716'))
)

In [30]:
class ParticipantVisibleError(Exception):
    pass


def cache_model(
    path: str | Path,
    exts: tuple[str, ...] = (".bin", ".pt", ".safetensors"),
    num_workers: int | None = None,
    chunk_mb: int = 256,
) -> int:
    """Pre-read model weight files into the OS page cache to speed up later loads.

    Args:
        path        : Directory containing model files, or a single file path.
        exts        : File extensions treated as model weight files.
        num_workers : Number of threads (default = min(CPU cores, 8)).
        chunk_mb    : Size of each read chunk in MB.

    Returns:
        Total bytes read (int).
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def warmup_file(fpath: Path) -> tuple[Path, int]:
        """Sequentially read an entire file in chunks."""
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        try:
            with open(fpath, "rb") as f:
                while True:
                    data = f.read(chunk_size)
                    if not data:
                        break
                    total += len(data)
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
        return fpath, total

    path = Path(path)
    # Collect files to read
    files: list[Path] = []
    if path.is_dir():
        files = [p for p in path.rglob("*") if p.is_file() and str(p).endswith(exts)]
        files.sort()
    else:
        files = [path] if path.exists() else []

    if not files:
        print(f"No model files found to cache at: {path}")
        return 0

    # Decide number of worker threads
    if num_workers is None:
        try:
            num_workers = min(multiprocessing.cpu_count(), 8)
        except Exception:
            num_workers = 4

    print(f"[cache_model] {len(files)} file(s), {num_workers} worker(s)")
    t0 = time.time()
    total_bytes = 0
    # Read files in parallel
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n
            print(f"[{i}/{len(files)}] cached {fpath.name}")

    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    speed = gb / elapsed if elapsed > 0 else 0
    print(f"[cache_model] total read ≈ {gb:.2f} GB")
    print(f"[cache_model] elapsed {elapsed:.2f} s, ~{speed:.2f} GB/s")
    return total_bytes


def extract_final_answer(text: str | None) -> str:
    r"""Extracts the final answer from the model response.

    Prioritizes extracting answers inside `\boxed{}`.
    If no `\boxed{}` format is found, attempts to extract numbers from other formats.

    Examples:
        >>> extract_final_answer(r"The answer is \boxed{42}")
        '42'
        >>> extract_final_answer("The final answer is: 3.14")
        '3.14'
        >>> extract_final_answer("Just a number 100 in text")
        '100'
        >>> extract_final_answer(None)
        'NOT_FOUND'
    """
    if text is None:
        return "NOT_FOUND"

    # Search for boxed answer
    # Match all instances of \boxed{...} or unclosed \boxed{ at the end
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    # Other common formats if \boxed{} is not found
    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
        r"final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    # If no structured format is found, extract the last valid number in the text
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if matches:
        return matches[-1]

    # If no numeric answer is found, return the last line of text as a fallback
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify(stored_answer: str, predicted: str) -> bool:
    """Verify if the answer matches.

    For numerical answers, allow them to be judged as equal within a certain relative tolerance (1e-2);
    otherwise, compare strictly as strings (case-insensitive).
    """
    # Clean up strings
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()

    try:
        # Try to convert the answers to floating point numbers
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        # Use a small absolute tolerance for numbers near zero
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        # Fallback to case-insensitive string comparison
        return predicted.lower() == stored_answer.lower()


def generate_standard_submission(submission_dir: str):
    """Processes an extracted submission archive to produce a standard submission file."""
    # Locate the LoRA files within the extracted directory
    possible_extraction_dirs = {
        "/kaggle/tmp",
        "/kaggle/working",
        submission_dir,
    }
    adapter_configs = []
    for search_dir in possible_extraction_dirs:
        if os.path.exists(search_dir):
            adapter_configs.extend(
                glob.glob(
                    os.path.join(search_dir, "**/adapter_config.json"), recursive=True
                )
            )
    if not adapter_configs:
        raise ParticipantVisibleError(
            "No adapter_config.json found in submission. Found:\n\n"
            f"{submission_dir} {os.listdir(submission_dir)}\n\n"
            f"/kaggle/tmp {os.listdir('/kaggle/tmp')}\n\n"
            f"/kaggle/input/competition_evaluation {os.listdir('/kaggle/input/competition_evaluation')}"
        )

    lora_path = os.path.dirname(adapter_configs[0])

    # Load test data
    test_df = pd.read_csv(DATA_PATH / "test.csv", index_col=None)

    row_id_col = str(test_df.columns.to_list()[0])
    predictions = []
    for item in test_df.itertuples(index=False):
        predictions.append(
            {
                row_id_col: getattr(item, row_id_col),
                "prediction": lora_path,
            }
        )

    submission_df = pd.DataFrame(predictions)

    # Write the standard submission file to the current working directory
    submission_df.to_csv("submission.csv", index=False)


def generate_predictions(
    test_df: pd.DataFrame,
    lora_path: str,
    row_id_col: str,
    max_lora_rank: int,
    max_tokens: int,
    top_p: float,
    temperature: float,
    max_num_seqs: int,
    gpu_memory_utilization: float,
    max_model_len: int,
    debug: bool = False,
) -> pd.DataFrame:
    """Load the model and generate predictions for the provided test data.

    Args:
        debug: If True, writes a CSV file with raw model outputs and extracted predictions.
    """
    # Cache Model
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["TRANSFORMERS_NO_FLAX"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    # Initialize vLLM Offline inference Engine
    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype="auto",
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )

    tokenizer = llm.get_tokenizer()
    prompts = []
    for item in test_df.itertuples(index=False):
        user_content = (
            item.prompt
            + "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
        )
        # Format using the tokenizer's chat template directly
        try:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            # Fallback if chat template fails
            prompt = user_content
        prompts.append(prompt)

    # Generate predictions using continuous batching
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest("adapter", 1, lora_path),
    )

    predictions = []
    debug_records = []
    for item, output in zip(test_df.itertuples(index=False), outputs):
        raw_text = output.outputs[0].text
        extracted_answer = extract_final_answer(raw_text)

        row_id_val = getattr(item, row_id_col)

        predictions.append(
            {
                row_id_col: row_id_val,
                "prediction": extracted_answer,
            }
        )

        if debug:
            debug_records.append(
                {
                    row_id_col: row_id_val,
                    "raw_output": raw_text,
                    "extracted_prediction": extracted_answer,
                }
            )

    # Write debug CSV if requested
    if debug and debug_records:
        debug_df = pd.DataFrame(debug_records)
        debug_df.to_csv("debug_predictions.csv", index=False)
        print("Debug data saved to debug_predictions.csv")

    return pd.DataFrame(predictions)


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    max_lora_rank: int = 32,
    max_tokens: int = 3584,
    top_p: float = 1.0,
    temperature: float = 1.0,
    max_num_seqs: int = 128,
    gpu_memory_utilization: float = 0.85,
    max_model_len: int = 4096,
    debug: bool = False,
) -> float:
    r"""Evaluate the generated predictions against the ground truth.

    Submissions are evaluated based on their **Accuracy** in solving the provided
    tasks. The NVIDIA Nemotron-3-Nano-30B model is loaded with the participant's
    submitted LoRA adapter (which must include an `adapter_config.json`) using
    the vLLM inference engine. For each test case, the model is prompted to
    generate a response and instructed to place its final answer within a `\boxed{}`
    LaTeX command. The metric extracts the final answer from the generated text,
    prioritizing content within the boxed format while falling back to other
    heuristic patterns or the first numeric value found. A prediction is graded as
    correct if it matches the ground truth either exactly as a string or within a
    relative numerical tolerance of $10^{-2}$. The final score is the proportion of
    correctly answered questions.

    Args:
        solution: DataFrame containing the ground truth answers. Must include the
            row_id_column_name and an 'answer' column.
        submission: DataFrame containing the predicted answers. Must include the
            row_id_column_name and a 'prediction' column.
        row_id_column_name: The name of the ID column used to join solution and
            submission.
        max_lora_rank: Maximum rank for LoRA adapters.
        max_tokens: Maximum number of tokens to generate.
        top_p: Top-p sampling parameter.
        temperature: Temperature sampling parameter.
        max_num_seqs: Maximum number of sequences to process concurrently.
        gpu_memory_utilization: Fraction of GPU memory to allocate for the vLLM execution.
        max_model_len: Maximum context length (input + output tokens).
        debug: If True, writes raw outputs and extracted predictions to a CSV file.

    Returns:
        The accuracy score (fraction of matches) as a float.
    """
    lora_path = submission["prediction"].iloc[0]

    # Load test data and filter it to only include rows present in the solution
    test_df = pd.read_csv(DATA_PATH / "test.csv", index_col=None)
    row_id_col = str(test_df.columns.to_list()[0])
    test_df = test_df[test_df[row_id_col].isin(solution[row_id_column_name])]

    submission = generate_predictions(
        test_df=test_df,
        lora_path=lora_path,
        row_id_col=row_id_column_name,
        max_lora_rank=max_lora_rank,
        max_tokens=max_tokens,
        top_p=top_p,
        temperature=temperature,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len,
        debug=debug,
    )

    dataset = solution.merge(submission, on=row_id_column_name)
    num_correct = 0

    # Verify the predictions
    for item in dataset.itertuples(index=False):
        ground_truth = item.answer
        extracted_answer = item.prediction

        match = verify(str(ground_truth), str(extracted_answer))
        if match:
            num_correct += 1

    accuracy = num_correct / len(solution)
    return float(accuracy)

In [31]:
# Cache Model
if TEST_GENERATION:
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

[cache_model] 13 file(s), 16 worker(s)
[1/13] cached model-00013-of-00013.safetensors
[2/13] cached model-00005-of-00013.safetensors
[3/13] cached model-00006-of-00013.safetensors
[4/13] cached model-00009-of-00013.safetensors
[5/13] cached model-00010-of-00013.safetensors
[6/13] cached model-00002-of-00013.safetensors
[7/13] cached model-00012-of-00013.safetensors
[8/13] cached model-00007-of-00013.safetensors
[9/13] cached model-00003-of-00013.safetensors
[10/13] cached model-00008-of-00013.safetensors
[11/13] cached model-00011-of-00013.safetensors
[12/13] cached model-00001-of-00013.safetensors
[13/13] cached model-00004-of-00013.safetensors
[cache_model] total read ≈ 58.82 GB
[cache_model] elapsed 65.05 s, ~0.90 GB/s


# Init vLLM

In [32]:
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

2026-04-13 07:15:17.861023: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776064518.058671      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776064518.116035      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776064518.634707      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776064518.634722      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776064518.634723      65 computation_placer.cc:177] computation placer alr

In [33]:
# www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/overview/evaluation
max_model_len = 8192
max_lora_rank = 32
max_tokens = 7680
top_p = 1.0
temperature = 0.0
max_num_seqs = 64
gpu_memory_utilization = 0.85
max_model_len = 8192

In [34]:
# Initialize vLLM Offline inference Engine

if TEST_GENERATION:
    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype="auto",
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )

INFO 04-13 07:15:52 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 32, 'enable_chunked_prefill': True, 'model': '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 04-13 07:16:24 [model.py:531] Resolved architecture: NemotronHForCausalLM
INFO 04-13 07:16:24 [model.py:1554] Using max model len 8192
INFO 04-13 07:16:24 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-13 07:16:24 [config.py:618] Updating mamba_ssm_cache_dtype to 'float32' for NemotronH model
WARNING 04-13 07:16:24 [config.py:381] Mamba cache mode is set to 'all' for NemotronHForCausalLM by default when prefix caching is enabled
INFO 04-13 07:16:24 [config.py:401] Warning: Prefix caching in Mamba cache 'all' mode is currently enabled. Its support for Mamba layers is experimental. Please report any issues you may observe.
INFO 04-13 07:16:24 [config.py:544] Setting attention block size to 2176 tokens to ensure that attention page size is >= mamba page size.
INFO 04-13 07:16:24 [config.py:575] Padding mamba page size by 4.41% to ensure that mamba page size and attention page size are exactly equal.
INFO 04-13 07:16:24 [vllm.py:747] Asynchron

2026-04-13 07:16:32.687441: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776064592.697733     454 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776064592.700853     454 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776064592.708676     454 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776064592.708694     454 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776064592.708695     454 computation_placer.cc:177] computation placer alr

(EngineCore_DP0 pid=454) INFO 04-13 07:16:43 [core.py:101] Initializing a V1 LLM engine (v0.17.1) with config: model='/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', speculative_config=None, tokenizer='/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityCon

...

(EngineCore_DP0 pid=454) WARNING 04-13 07:17:03 [fused_moe.py:1093] Using default MoE config. Performance might be sub-optimal! Config file not found at /tmp/vllm/model_executor/layers/fused_moe/configs/E=128,N=1856,device_name=NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition.json
(EngineCore_DP0 pid=454) INFO 04-13 07:17:09 [backends.py:366] Compiling a graph for compile range (1, 16384) takes 10.55 s
(EngineCore_DP0 pid=454) INFO 04-13 07:17:09 [monitor.py:35] torch.compile takes 14.11 s in total
(EngineCore_DP0 pid=454) INFO 04-13 07:17:09 [decorators.py:580] saving AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/a745ba0e914f72dd8b769997a9d9c6e0e60091cdac8cb4325b93233c9af338a2/rank_0_0/model
(EngineCore_DP0 pid=454) INFO 04-13 07:17:09 [decorators.py:588] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/a745ba0e914f72dd8b769997a9d9c6e0e60091cdac8cb4325b93233c9af338a2/rank_0_0/model
(EngineCore_DP0 pid=454) INFO 04-13

(EngineCore_DP0 pid=454) 2026-04-13 07:17:14,952 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=454) 2026-04-13 07:17:15,153 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/38 [00:00<?, ?it/s]

(EngineCore_DP0 pid=454) WARNING 04-13 07:17:15 [utils.py:268] Using default LoRA kernel configs


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 38/38 [00:11<00:00,  3.19it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 22/22 [00:41<00:00,  1.87s/it]


(EngineCore_DP0 pid=454) INFO 04-13 07:18:09 [gpu_model_runner.py:5386] Graph capturing finished in 54 secs, took -3.30 GiB
(EngineCore_DP0 pid=454) INFO 04-13 07:18:09 [core.py:282] init engine (profile, create kv cache, warmup model) took 74.97 seconds
(EngineCore_DP0 pid=454) INFO 04-13 07:18:10 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 04-13 07:18:10 [llm.py:388] Supported tasks: ['generate']


# Test generation

In [35]:
import pandas as pd
df = pd.read_csv("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv")

problem_set = {
    # bit_manipulation
    "836b85e8", "b20b39bf", "af358750", "3f9bd1e7", "0528d502", "9992bbd0", "812131f1", "84e3f9f7",  # min_lp: -20.0965 .. -18.6948
    "114a41e3", "585f2ff6", "ea6859d7", "2fa48efe", "5f76ba09", "6a186446", "5d0db0d2", "75898981",  # min_lp: -18.6263 .. -17.7624
    "c7a37cda", "989dde0a", "d623e937", "100e280a", "f6a95641", "302dc36e", "3c9b8e0e", "b6e4a36d",  # min_lp: -17.6909 .. -17.3764
    "bd214050", "d50683b4", "f7346f0c", "5dd3345c", "dfc4839c", "0e7a6920", "31a4c9ef", "f8fc43d2",  # min_lp: -17.3617 .. -16.6266
    "19f4b3d6", "093de4ea", "9bd65991", "c6fa3e3f", "b8e0c853", "cb3317fe", "4ada9150", "c90fa3a6",  # min_lp: -16.5006 .. -15.4493
    # cipher
    "cf821623", "31c72d27", "73cd9008", "3975d230", "4db54201", "c1ffb3ac", "2a6c343e", "0e46fd1d",  # min_lp: -19.4908 .. -18.5764
    "49b244e3", "7b8e4432", "3019f44e", "4f8f23d6", "1fcbbb93", "987a223b", "84d10c70", "0dad87bf",  # min_lp: -18.5705 .. -18.2567
    # cryptarithm_deduce
    "1c7a0091", "ed61a9d6", "02b8d816", "d6c03e21", "2f6531cb", "a4ee9fa6", "02a04b59", "81b6d789",  # min_lp: -16.8100 .. -8.6917
    "bc83b0a1", "b1b10e83", "dea42835", "2c017f70", "b13d511a", "6897f05e", "64d775e5", "24e1f1d5",  # min_lp: -8.1260 .. -3.0486
    # cryptarithm_guess
    "c7844441", "0fcf912a", "07b440f0", "25ee72c3", "9dfe5ac9", "deed3497", "258b796b", "0da1841f",  # min_lp: -12.5001 .. -0.1081
    "e38c423b", "2e9973b7", "55f4fa64",  # min_lp: -0.0486 .. -0.0202
    # equation_numeric_deduce
    "91488dc9", "7e2e8a95", "35a89469", "a04ecffd", "27cec7a9", "d6a2e332", "04322d27", "8bc6a26c",  # min_lp: -21.2890 .. -17.3401
    "5c743e8a", "30763ac0", "91b42a45", "fecad63c", "c857a727", "b69391b8", "45df54db", "e5956ffa",  # min_lp: -17.0230 .. -15.1098
    # equation_numeric_guess
    "662fd21c", "e9e6b620", "8ae8e12a", "dc178d1c", "9c91b226", "c763054a", "66a0856f", "ef1b13ac",  # min_lp: -19.6251 .. -2.2399
    "8df3daad", "e7b87b82", "5d89a09c", "be877da5", "7c0c5227", "69fe4b0d", "b7ad0671", "4e840a1a",  # min_lp: -2.1279 .. -0.1560
    # gravity
    "d0dd2df7", "74a50b2c", "f9f20a7a", "ef3c7703", "85bc954c", "8de7d8bc", "22bb13b8", "87eb7ce0",  # min_lp: -18.8125 .. -16.7516
    "8a24aef9", "3c53c8af", "a6f1b553", "7b47f88d", "c0f6e1b8", "44dbe7d3", "827a6b1b", "6f3a0625",  # min_lp: -16.6875 .. -15.1253
    # numeral
    "3b4ebafd", "ad6ff612", "5a6ed2bf", "0adca57b", "d79d0cfd", "e6b04620", "f47276a4", "1e2de753",  # min_lp: -18.7504 .. -12.9401
    "0122d53a", "685bb0b1", "797ae611", "588a4ce8", "f19ffbf1", "8c281ee9", "972ef18a", "5092f0e0",  # min_lp: -12.4973 .. -11.1420
    # unit_conversion
    "082c1a06", "3dcaf042", "87342969", "8e1cff16", "d566ff0e", "598af975", "51a22965", "d3d82844",  # min_lp: -22.2500 .. -20.1875
    "e6157d05", "cd1280b0", "bbb61c3a", "740e0460", "be2416ec", "63ec749f", "26e6819a", "99948ad9",  # min_lp: -19.7500 .. -17.4077
}
df = df[df.id.isin(problem_set)].copy()

In [36]:
problem_texts = list(df["prompt"])

if TEST_GENERATION:
    tokenizer = llm.get_tokenizer()
    prompts = []
    for problem_text in problem_texts:
        # Format using the tokenizer's chat template directly
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": problem_text}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        prompts.append(prompt)

In [37]:
# Generate predictions using continuous batching (without adapter, for baseline comparison)
if TEST_GENERATION:
    # outputs = llm.generate(
    #     prompts,
    #     sampling_params=sampling_params,
    # )
    pass

In [38]:
print(os.listdir("."))

['reference', '__notebook__.ipynb', 'adapter_config.json', 'checkpoint_complete', 'adapter_model.safetensors', 'README.md']


In [39]:
possible_extraction_dirs = {
    "/kaggle/tmp",
    "/kaggle/working",
    # submission_dir,
}
adapter_configs = []
for search_dir in possible_extraction_dirs:
    if os.path.exists(search_dir):
        adapter_configs.extend(
            glob.glob(
                os.path.join(search_dir, "**/adapter_config.json"), recursive=True
            )
        )

lora_path = os.path.dirname(adapter_configs[0])

In [40]:
print(os.listdir("/kaggle/working"))

['reference', '__notebook__.ipynb', 'adapter_config.json', 'checkpoint_complete', 'adapter_model.safetensors', 'README.md']


In [41]:
with open("adapter_config.json") as f:
    print(f.read())

{"alpha_pattern": {}, "auto_mapping": null, "base_model_name_or_path": null, "bias": "none", "corda_config": null, "eva_config": null, "exclude_modules": null, "fan_in_fan_out": false, "inference_mode": false, "init_lora_weights": true, "layer_replication": null, "layers_pattern": null, "layers_to_transform": null, "loftq_config": {}, "lora_alpha": 32, "lora_bias": false, "lora_dropout": 0, "megatron_config": null, "megatron_core": "megatron.core", "modules_to_save": null, "peft_type": "LORA", "r": 32, "rank_pattern": {}, "revision": null, "target_modules": ["k_proj", "o_proj", "in_proj", "q_proj", "up_proj", "v_proj", "down_proj", "out_proj", "lm_head"], "task_type": "CAUSAL_LM", "trainable_token_indices": null, "use_dora": false, "use_rslora": false}


In [42]:
print(lora_path)

/kaggle/working


In [43]:
# Generate predictions using continuous batching (with adapter)
if TEST_GENERATION:
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest("adapter", 1, lora_path),
    )
    df["output"] = [output.outputs[0].text for output in outputs]
    df["predicted"] = df["output"].apply(extract_final_answer)
    df["correct"] = df.apply(lambda row: verify(str(row["answer"]), str(row["predicted"])), axis=1)

Rendering prompts:   0%|          | 0/163 [00:00<?, ?it/s]

WARNING 04-13 07:18:10 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/163 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

# Produce submission

In [44]:
import zipfile as _zf

print(os.listdir("."))
shutil.rmtree("reference", ignore_errors=True)
with _zf.ZipFile("submission.zip", "w", _zf.ZIP_DEFLATED) as zf:
    for file in os.listdir("."):
        if file.startswith(".") or file == "submission.zip" or not os.path.isfile(file):
            continue
        zf.write(file)
        os.remove(file)

['reference', '__notebook__.ipynb', 'adapter_config.json', 'checkpoint_complete', 'adapter_model.safetensors', 'README.md']


In [45]:
df.to_csv("predictions.csv", index=False)

In [46]:
print(os.listdir("."))

['submission.zip', 'predictions.csv', '__notebook__.ipynb']
